# Transformers & Prompt Engineering — Hands-On Session
**Duration:** ~1 hour | **Level:** Beginner | **Runtime:** Google Colab (GPU) | **Model:** Gemma 4 (offline)

---

## Agenda
| # | Topic | Time |
|---|-------|------|
| 1 | What is a Transformer? | 10 min |
| 2 | How LLMs generate text | 5 min |
| 3 | Setup: Load Gemma 4 locally in Colab | 5 min |
| 4 | Prompt Engineering Techniques | 25 min |
| 5 | Exercises | 15 min |

> **Before starting:** Runtime → Change runtime type → **T4 GPU**

---
## Part 1 — What is a Transformer? (High-Level Intuition)

### The Problem Before Transformers

Before 2017, AI models processed text **one word at a time** (like reading a book one letter at a time — slow and forgetful).

```
Old approach (RNN):  word1 → word2 → word3 → word4  (sequential, slow)
Transformer:         word1 ↘
                     word2 → [looks at ALL words at once] → output
                     word3 ↗
```

### The Big Idea: ATTENTION

Transformers introduced **"Attention"** — the ability to focus on relevant words regardless of their position.

**Real example:**
> *"The animal didn't cross the street because **it** was too tired."*

What does **"it"** refer to? The animal! A Transformer figures this out by paying **attention** to the right word.

```
"it"  pays attention to →  "animal" (high score)
                        →  "street" (low score)
                        →  "tired"  (medium score)
```

### Key Components (Simplified)

```
INPUT TEXT
    ↓
[Tokenizer]      — splits text into tokens (words/subwords)
    ↓
[Embeddings]     — converts tokens to numbers (vectors)
    ↓
[Attention]      — each token looks at all other tokens
    ↓
[Feed Forward]   — processes the attended information
    ↓
    × N layers   — repeat attention + feedforward N times
    ↓
OUTPUT (next token prediction)
```

### Tokenization — What the Model Actually Sees

```
"Hello, I love transformers!"
        ↓
["Hello", ",", "I", "love", "transform", "##ers", "!"]
        ↓
[  7592,  1010, 1045, 2293,   10938,      2545,  999  ]
```

The model sees **numbers**, not words!

### Famous Transformer Models

| Model | Creator | Use |
|-------|---------|-----|
| GPT-4 | OpenAI | Chat, reasoning |
| Claude | Anthropic | Chat, analysis |
| **Gemma** | **Google** | **Open, efficient, fast** |
| Llama 3 | Meta | Open source LLM |
| BERT | Google | Understanding text |

> **Key Takeaway:** A Transformer is a neural network that reads all words simultaneously and learns which words to pay attention to for a given task.

> **About Gemma:** Gemma is Google's family of open-weight Transformer models. We're using it today via Hugging Face — it runs in the cloud, no GPU needed!

---
## Part 2 — How LLMs Generate Text

LLMs are **next-token predictors**. They predict the most likely next word, one at a time.

```
Prompt:  "The capital of France is"
Model:   → "Paris"  (probability: 95%)
         → "Lyon"   (probability: 3%)
         → "Berlin" (probability: 0.5%)
```

Then it appends "Paris" and predicts again:
```
"The capital of France is Paris" → "."
```

### Temperature — Creativity Knob

| Temperature | Behavior | Use case |
|------------|----------|----------|
| 0.0 | Always picks most likely word | Facts, code |
| 0.7 | Balanced | General chat |
| 1.0+ | More random/creative | Storytelling, brainstorming |

> **This means:** The model has NO memory, NO understanding — it just predicts tokens extremely well. **Your prompt is everything.**

---
## Part 3 — Setup: Load Gemma 4 Locally in Colab

We load Gemma 4 **directly into Colab's GPU** — no API calls, fully offline after download.

**Steps:**
1. In Colab: **Runtime → Change runtime type → T4 GPU** (free tier works)
2. Get a HF token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → Read token
3. Accept Gemma 4 license: [huggingface.co/google/gemma-4-4b-it](https://huggingface.co/google/gemma-4-4b-it)
4. Run the cells below — model downloads once (~3 GB), then runs offline

**Model:** `google/gemma-4-4b-it` with **4-bit quantization**

| Gemma 4 variant | VRAM needed | Colab tier |
|-----------------|-------------|------------|
| `gemma-4-4b-it` + 4-bit | ~4 GB | Free T4 ✅ |
| `gemma-4-12b-it` + 4-bit | ~8 GB | Free T4 ✅ |
| `gemma-4-27b-it` + 4-bit | ~18 GB | Colab Pro (A100) |

> **4-bit quantization** = compresses the model weights to use less GPU memory with minimal quality loss.

In [ ]:
# Install required libraries (run once)
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import pipeline, BitsAndBytesConfig

HF_TOKEN = "hf_your_token_here"  # Replace with your token
MODEL    = "google/gemma-4-4b-it"

# 4-bit quantization — fits the model into free Colab T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL}... (downloads ~3 GB on first run, then cached)")

pipe = pipeline(
    "text-generation",
    model=MODEL,
    model_kwargs={"quantization_config": bnb_config},
    device_map="auto",
    token=HF_TOKEN,
)

print("Model loaded!")

def ask(prompt, system="You are a helpful assistant.", temperature=0.7, max_tokens=300):
    """Run inference locally with Gemma 4 on Colab GPU."""
    full_prompt = f"{system}\n\n{prompt}" if system else prompt

    messages = [{"role": "user", "content": full_prompt}]

    output = pipe(
        messages,
        max_new_tokens=max_tokens,
        temperature=temperature,
        do_sample=temperature > 0,
    )
    result = output[0]["generated_text"][-1]["content"]
    print(result)
    return result

# Test
ask("Say 'Gemma 4 is running locally on Colab!' and nothing else.")

---
## Part 4 — Prompt Engineering Techniques

> **Prompt Engineering** = The art of crafting inputs to get the best outputs from an LLM.

We'll cover **5 core techniques:**
1. Zero-Shot Prompting
2. Few-Shot Prompting
3. Chain-of-Thought (CoT)
4. Role / System Prompting
5. Output Formatting

### Technique 1 — Zero-Shot Prompting

Give the model a task with **no examples**. Just tell it what to do.

```
Pattern:
  [Task description] + [Input]
```

In [ ]:
# DEMO 1A: Bad zero-shot (vague)
print("=== BAD Prompt ===")
ask("Tell me about this: The movie was amazing but too long.")

In [ ]:
# DEMO 1B: Good zero-shot (clear task)
print("=== GOOD Prompt ===")
ask("""
Classify the sentiment of the following movie review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The movie was amazing but too long."

Sentiment:
""")

**Lesson:** Be explicit. Tell the model exactly what output you want.

---
### Technique 2 — Few-Shot Prompting

Give the model **2–5 examples** of input → output before the real question.

```
Pattern:
  [Example 1: input → output]
  [Example 2: input → output]
  [Example 3: input → output]
  [Your actual input → ???]
```

Think of it as: *"Here's how I want you to respond. Now do the same for this."*

In [ ]:
# DEMO 2A: Zero-shot (model guesses the format)
print("=== Zero-Shot (no examples) ===")
ask("Convert this to formal English: gonna grab some grub later")

In [ ]:
# DEMO 2B: Few-shot (model follows the pattern)
print("=== Few-Shot (with examples) ===")
ask("""
Convert informal English to formal English.

Informal: "wassup, u free tmrw?"
Formal: "Good day, are you available tomorrow?"

Informal: "omg that meeting was super boring lol"
Formal: "The meeting was quite unengaging."

Informal: "gonna grab some grub later"
Formal:
""")

**Lesson:** Examples teach the model the **format, tone, and style** you expect — without writing long instructions.

In [ ]:
# DEMO 2C: Few-shot for custom classification
ask("""
Classify customer support tickets into: BILLING, TECHNICAL, SHIPPING, or OTHER.

Ticket: "I was charged twice for my order"
Category: BILLING

Ticket: "My app keeps crashing on startup"
Category: TECHNICAL

Ticket: "My package hasn't arrived in 2 weeks"
Category: SHIPPING

Ticket: "How do I reset my password?"
Category:
""")

---
### Technique 3 — Chain-of-Thought (CoT)

Ask the model to **think step by step** before giving the final answer.

```
Pattern:
  [Question] + "Think step by step."
  
  OR show a step-by-step example first (few-shot CoT)
```

This dramatically improves reasoning on **math, logic, and multi-step problems**.

In [ ]:
# DEMO 3A: Without CoT — model may rush to wrong answer
print("=== WITHOUT Chain-of-Thought ===")
ask("""
A store has 5 boxes. Each box has 4 bags. Each bag has 3 apples. 
If you remove 10 apples, how many are left?

Answer:
""", temperature=0.1)

In [ ]:
# DEMO 3B: With CoT — model reasons through it
print("=== WITH Chain-of-Thought ===")
ask("""
A store has 5 boxes. Each box has 4 bags. Each bag has 3 apples. 
If you remove 10 apples, how many are left?

Let's think step by step:
""", temperature=0.1)

In [ ]:
# DEMO 3C: CoT for decision making
ask("""
Should I use a SQL database or MongoDB for a social media app 
where users post photos and follow each other?

Think through the requirements step by step, then give a recommendation.
""", temperature=0.3, max_tokens=400)

**Lesson:** Adding *"think step by step"* is one of the easiest wins in prompt engineering. It forces the model to reason before committing to an answer.

---
### Technique 4 — Role / System Prompting

Give the model a **persona** or **role** to shape its entire behavior.

```
Pattern:
  System: "You are a [role] who [constraints/style]."
  User:   [your actual question]
```

The system prompt sets the **personality, expertise, and rules** for the whole conversation.

In [ ]:
# DEMO 4A: Default assistant
print("=== Default Assistant ===")
ask("What is recursion?")

In [ ]:
# DEMO 4B: Same question, different role = different response
print("=== Expert Python Teacher for Kids ===")
ask(
    prompt="What is recursion?",
    system="""
    You are a friendly Python teacher explaining concepts to 10-year-old children.
    Always use simple words, fun analogies (like food, games, or animals), 
    and short sentences. Never use technical jargon.
    """
)

In [ ]:
# DEMO 4C: Role with strict constraints (useful for production apps)
print("=== Customer Support Bot ===")
ask(
    prompt="Can you write me a poem?",
    system="""
    You are a customer support agent for TechShop, an electronics store.
    You ONLY answer questions about:
    - Product availability and pricing
    - Order status and returns
    - Technical support for TechShop products
    
    For ANY other topic, politely say it's outside your scope 
    and redirect to the relevant topic.
    """
)

**Lesson:** System prompts let you **customize the model's identity** for your specific application. This is how products like customer bots, tutors, and coding assistants are built.

---
### Technique 5 — Output Formatting

Tell the model **exactly how to format its response** — JSON, bullet points, table, markdown, etc.

```
Pattern:
  [Task] + "Return your answer as [format]."
  
  OR provide a template/schema to fill.
```

Critical for when the output feeds into another system (database, UI, API).

In [ ]:
# DEMO 5A: Unformatted (hard to parse programmatically)
print("=== No Format Specified ===")
ask("Give me info about Python the programming language.")

In [ ]:
# DEMO 5B: JSON output (easy to parse in code)
print("=== Structured JSON Output ===")
ask("""
Extract information about Python the programming language.

Return ONLY valid JSON with this exact structure, no extra text:
{
  "name": "",
  "created_year": 0,
  "creator": "",
  "top_use_cases": [],
  "is_open_source": true
}
""", temperature=0.1)

In [ ]:
# DEMO 5C: Combining techniques — Role + Few-shot + Format
print("=== Combined Techniques ===")
ask(
    system="You are an expert product reviewer who writes concise, honest reviews.",
    prompt="""
Review the following product and return output in this exact markdown format:

**Product:** [name]
**Rating:** [X/5 stars]
**Pros:**
- [pro 1]
- [pro 2]
**Cons:**
- [con 1]
**Verdict:** [one sentence]

Product: MacBook Air M2
""",
    temperature=0.3
)

---
## Quick Reference — Prompt Engineering Cheat Sheet

| Technique | When to Use | Magic Words |
|-----------|-------------|-------------|
| **Zero-Shot** | Simple, clear tasks | `"Classify...", "Summarize...", "Translate..."` |
| **Few-Shot** | Custom format / style needed | Show 2-5 examples |
| **Chain-of-Thought** | Math, logic, multi-step reasoning | `"Think step by step"` |
| **Role Prompting** | Consistent persona or constraints | `"You are a..."` (system prompt) |
| **Output Format** | Structured output for code/APIs | `"Return as JSON", "Use this template"` |

### Golden Rules
1. **Be specific** — vague prompts = vague answers
2. **Show, don't just tell** — examples beat long instructions
3. **Control the format** — if you need structured output, specify it
4. **Iterate** — prompt engineering is trial and error
5. **Lower temperature** for facts/code, **higher** for creativity

---
## Part 5 — EXERCISES

Try these yourself! Refer to the techniques above.

> **Tip:** There's no single correct answer. Experiment and see what works best.

### Exercise 1 — Zero-Shot (Easy)

Write a zero-shot prompt to detect the **language** of a given text.
Test it with: `"Bonjour, comment ça va?"`

Expected output format: just the language name, e.g. `French`

In [ ]:
# YOUR CODE HERE
ask("""
TODO: Write your zero-shot prompt here

Text: "Bonjour, comment ça va?"
Language:
""")

### Exercise 2 — Few-Shot (Medium)

Build a few-shot prompt that converts temperatures from Celsius to a **human-friendly description**.

Example outputs you want:
- `0°C` → `"Freezing cold — wear a heavy coat"`
- `22°C` → `"Perfectly comfortable — light clothing works"`
- `38°C` → `"Extremely hot — stay hydrated"`

Then test it with: `15°C` and `45°C`

In [ ]:
# YOUR CODE HERE
ask("""
TODO: Add your few-shot examples here, then put 15°C and 45°C as the test inputs

Temperature: 0°C
Description: "Freezing cold — wear a heavy coat"

Temperature: 22°C
Description: "Perfectly comfortable — light clothing works"

Temperature: 38°C
Description: "Extremely hot — stay hydrated"

Temperature: 15°C
Description:
""")

### Exercise 3 — Chain-of-Thought (Medium)

Use CoT to solve this logic puzzle:

> Alice, Bob, and Carol each have a pet: a dog, a cat, and a fish (in some order).
> - Alice does not have the fish.
> - Bob does not have the cat.
> - Carol does not have the dog.
>
> Who has which pet?

Write a prompt that makes the model reason step by step.

In [ ]:
# YOUR CODE HERE
ask("""
TODO: Add your Chain-of-Thought prompt here

Alice, Bob, and Carol each have a pet: a dog, a cat, and a fish (in some order).
- Alice does not have the fish.
- Bob does not have the cat.
- Carol does not have the dog.

Who has which pet? (Add CoT instruction here)
""", temperature=0.1)

### Exercise 4 — Role + Format (Hard)

Build a prompt that acts as a **code reviewer**.

Requirements:
- System role: senior Python developer, strict about clean code
- Input: a Python function (provided below)
- Output format: JSON with keys: `issues` (list), `score` (1-10), `improved_code` (string)

Test with this function:
```python
def calc(x,y,op):
    if op=="add": return x+y
    if op=="sub": return x-y
    if op=="mul": return x*y
    if op=="div": return x/y
```

In [ ]:
# YOUR CODE HERE
code = """
def calc(x,y,op):
    if op=="add": return x+y
    if op=="sub": return x-y
    if op=="mul": return x*y
    if op=="div": return x/y
"""

ask(
    system="TODO: Write your role/system prompt here",
    prompt=f"""TODO: Write your prompt with format requirements here\n\nCode:\n{code}""",
    temperature=0.2,
    max_tokens=500
)

### Exercise 5 — BONUS: Build a Mini App

Combine everything you learned to build a **personal email assistant**.

Requirements:
1. System role: professional email writer
2. Takes a **rough note** (informal, incomplete) as input
3. Outputs a **polished email** with: Subject line, Greeting, Body, Sign-off
4. Test with: `"tell john meeting moved to friday 3pm, pls confirm, also bring the q3 report"`

In [ ]:
# YOUR CODE HERE
rough_note = "tell john meeting moved to friday 3pm, pls confirm, also bring the q3 report"

ask(
    system="TODO: Your system prompt",
    prompt=f"TODO: Your prompt using the rough_note variable: {rough_note}",
    temperature=0.4,
    max_tokens=400
)

---
## Solutions (Peek After Trying!)

Run the cell below to reveal solutions.

In [ ]:
solutions = {
    "Ex1 - Zero-Shot": """
Identify the language of the following text. Reply with ONLY the language name.

Text: "Bonjour, comment ça va?"
Language:
""",
    "Ex3 - CoT magic words": "'Let's work through this step by step:'",
    "Ex4 - System prompt": "You are a senior Python developer who reviews code strictly for readability, correctness, and best practices. Be direct and specific.",
    "Ex5 - System prompt": "You are a professional email writer. Convert rough notes into polished, concise business emails. Always include a subject line, greeting, clear body, and sign-off."
}

for exercise, solution in solutions.items():
    print(f"{'='*50}")
    print(f"🔑 {exercise}")
    print(solution)

---
## Summary — What You Learned Today

### Transformers
- Process ALL words simultaneously using **Attention**
- Convert text → tokens → numbers for processing
- Predict the next token, one at a time
- **Temperature** controls creativity vs. determinism

### Prompt Engineering

| Technique | Key Idea |
|-----------|----------|
| Zero-Shot | Be specific and explicit about the task |
| Few-Shot | Give examples to teach format and style |
| Chain-of-Thought | "Think step by step" for complex reasoning |
| Role Prompting | System prompt shapes persona and constraints |
| Output Format | Specify exact structure for programmatic use |

### Next Steps
- Try **RAG** (Retrieval-Augmented Generation) — give the model your own documents
- Explore **function calling** — let the model trigger your code
- Learn about **fine-tuning** — train the model on your specific data

---
*Running Google Gemma 4 (`google/gemma-4-4b-it`) offline via Hugging Face Transformers on Google Colab*